# Scored XLSX to JSON

In [9]:
import pandas as pd
import json
import uuid
from datetime import date
import json
import numpy as np
from shapely.geometry import Point, LineString
import requests
import polyline

SCORED_PATH = "../data/scored/candidates_scored.xlsx"
OUTPUT_PATH = "../data/production/restaurants.json"

df = pd.read_excel(SCORED_PATH)
print(f"{len(df)} restaurants geladen")
df_approved = df[df["qualifies"] == 1].copy()
print(f"{len(df_approved)} goedgekeurde restaurants")

62 restaurants geladen
21 goedgekeurde restaurants


## Defining functions

In [8]:
def get_route(waypoints):
    coords = ";".join([f"{lng},{lat}" for lat, lng in waypoints])
    url = f"http://router.project-osrm.org/route/v1/driving/{coords}"
    params = {"overview": "full", "geometries": "polyline"}
    response = requests.get(url, params=params).json()
    encoded = response["routes"][0]["geometry"]
    return polyline.decode(encoded)

def compute_km_marker(lat, lng, route_coords):
    """
    Zoek het dichtstbijzijnde punt op de route
    en bereken de afstand langs de route tot dat punt.
    """
    min_dist = float("inf")
    closest_idx = 0

    for i, (rlat, rlng) in enumerate(route_coords):
        dist = (lat - rlat) ** 2 + (lng - rlng) ** 2
        if dist < min_dist:
            min_dist = dist
            closest_idx = i

    # Bereken afstand langs de route tot closest_idx
    km = 0.0
    for i in range(1, closest_idx + 1):
        lat1, lng1 = route_coords[i - 1]
        lat2, lng2 = route_coords[i]
        # Haversine benadering: 1 graad lat ≈ 111km, 1 graad lng ≈ 80km op deze breedtegraad
        dlat = (lat2 - lat1) * 111
        dlng = (lng2 - lng1) * 80
        km += (dlat ** 2 + dlng ** 2) ** 0.5

    return round(km, 1)

def get_detour_minutes(rest_lat, rest_lng, route_coords):
    """
    Zoek dichtstbijzijnde punt op de route (proxy voor afslag)
    en bereken rijtijd van daar naar het restaurant via OSRM.
    """
    min_dist = float("inf")
    closest = None
    for rlat, rlng in route_coords:
        dist = (rest_lat - rlat) ** 2 + (rest_lng - rlng) ** 2
        if dist < min_dist:
            min_dist = dist
            closest = (rlat, rlng)

    exit_lat, exit_lng = closest
    url = (f"http://router.project-osrm.org/route/v1/driving/"
           f"{exit_lng},{exit_lat};{rest_lng},{rest_lat}")
    params = {"overview": "false"}
    response = requests.get(url, params=params).json()
    duration_sec = response["routes"][0]["duration"]
    return round(duration_sec / 60, 1)

def build_record(row):
    return {
        "id":             str(uuid.uuid4()),
        "place_id":       row["place_id"],
        "name":           row["name"],
        "address":        row["address"],
        "lat":            row["lat"],
        "lng":            row["lng"],
        "km_marker":      None,  # handmatig in te vullen
        "detour_minutes": None,  # handmatig in te vullen
        "priority":       int(row["priority"]),
        "price_range":    int(row["price_level"]) if pd.notna(row["price_level"]) else None,
        "editorial_note": None,  # in te vullen na bezoek/bel
        "phone":          None,  # in te vullen
        "website":        None,  # in te vullen
        "opening_hours":  None,  # in te vullen na bellen
        "google_maps_url": row["google_maps_url"],
        "verified_date":  None,
        "active":         True,
    }


# Building the JSON file, enrichting 

In [10]:
restaurants_json = [build_record(row) for _, row in df_approved.iterrows()]

waypoints = [
    (52.39, 4.64),   # Haarlem
    (48.85, 2.35),   # Paris
]

route_coords = get_route(waypoints)

for r in restaurants_json:
    r["km_marker"] = compute_km_marker(r["lat"], r["lng"], route_coords)
    r["detour_minutes"] = get_detour_minutes(r["lat"], r["lng"], route_coords)
    print(f"{r['name']:<50} km {r['km_marker']} and {r['detour_minutes']} min detour")

Restaurant Beukenhof Vichte                        km 282.2 and 7.8 min detour
't Brigandje                                       km 262.5 and 38.1 min detour
't Veer                                            km 270.7 and 23.4 min detour
Vijverhof                                          km 299.1 and 14.1 min detour
Ferme Balthazar                                    km 293.6 and 10.4 min detour
Le Pavé Gourmand                                   km 324.1 and 19.9 min detour
RESTAURANT LE STROMBOLI -RESTAURANT-PIZZÉRIA BAPAUME ET ENVIRONS km 386.5 and 26.4 min detour
Auberge de la Vallée d'Ancre                       km 389.1 and 32.7 min detour
Aux Gars du Nord                                   km 401.1 and 20.1 min detour
Le Méditerranée Sarl                               km 405.8 and 17.6 min detour
À la Chouette Gourmande                            km 406.5 and 14.1 min detour
La Taverne du Cochon Salé                          km 402.5 and 36.8 min detour
Le Bistrot D'Antoine       

In [10]:
import os
os.makedirs("../data/processed", exist_ok=True)

def build_record(row):
    return {
        "id":             str(uuid.uuid4()),
        "place_id":       row["place_id"],
        "name":           row["name"],
        "address":        row["address"],
        "lat":            row["lat"],
        "lng":            row["lng"],
        "km_marker":      None,  # handmatig in te vullen
        "detour_minutes": None,  # handmatig in te vullen
        "priority":       int(row["priority"]),
        "price_range":    int(row["price_level"]) if pd.notna(row["price_level"]) else None,
        "editorial_note": None,  # in te vullen na bezoek/bel
        "phone":          None,  # in te vullen
        "website":        None,  # in te vullen
        "opening_hours":  None,  # in te vullen na bellen
        "google_maps_url": row["google_maps_url"],
        "verified_date":  None,
        "active":         True,
    }

records = [build_record(row) for _, row in df_approved.iterrows()]

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"Opgeslagen: {len(records)} restaurants → {OUTPUT_PATH}")

Opgeslagen: 20 restaurants → ../data/production/restaurants.json
